# Plotly Interactive Visualisation

## Overview

Plotly produces interactive, browser-based charts. It is the standard choice when interactivity aids investigation.

| Interface | Use when |
|---|---|
| `plotly.express` (px) | Standard chart types quickly |
| `plotly.graph_objects` (go) | Custom layouts, mixed trace types |

---

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

rng = np.random.default_rng(42)
n = 300
elevation = rng.uniform(50, 400, n)
sites = pd.DataFrame({
    "site_id":    [f"SITE_{i:03d}" for i in range(1, n+1)],
    "catchment":  rng.choice(["North", "South", "East", "West"], n),
    "elevation":  elevation.round(1),
    "nitrate":    (rng.gamma(2, 2, n) + 0.008*elevation).round(2),
    "phosphorus": (rng.gamma(1.5, 1.5, n) + 0.004*elevation).round(2),
    "ph":         (rng.normal(7.5, 0.3, n) - 0.002*elevation).round(2),
    "treatment":  rng.choice(["control", "restored"], n),
})
sites["richness"] = list(map(int, (20 - 0.03*elevation + rng.normal(0,4,n)).round().clip(2,40)))
print(sites.shape)

(300, 8)


---
## Scatter with Hover and Trendline

In [2]:
fig = px.scatter(
    sites, x="elevation", y="richness",
    color="catchment", symbol="treatment",
    hover_data={"site_id": True, "nitrate": ":.2f", "ph": ":.2f"},
    trendline="ols",
    title="Species Richness vs. Elevation",
    labels={"elevation": "Elevation (m)", "richness": "Species richness"},
    template="plotly_white"
)
fig.update_traces(marker=dict(size=6, opacity=0.65))
fig.show()

---
## Distribution Plots

In [3]:
fig = px.histogram(
    sites, x="nitrate", color="treatment",
    marginal="box", barmode="overlay", opacity=0.55, nbins=40,
    title="Nitrate by Treatment", template="plotly_white"
)
fig.show()
fig2 = px.box(
    sites, x="catchment", y="richness", color="treatment",
    points="outliers", title="Richness by Catchment",
    template="plotly_white"
)
fig2.show()

---
## Multi-Panel with graph_objects

In [4]:
fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Nitrate vs. Richness", "Elevation by Catchment"))
c = {"North":"#4a8fff","South":"#4fffb0","East":"#ff6b6b","West":"#ffd166"}
for cat, grp in sites.groupby("catchment"):
    fig.add_trace(go.Scatter(
        x=grp["nitrate"], y=grp["richness"], mode="markers", name=cat,
        marker=dict(color=c[cat], size=5, opacity=0.5)), row=1, col=1)
    fig.add_trace(go.Histogram(
        x=grp["elevation"], name=cat, opacity=0.5, nbinsx=25,
        marker_color=c[cat], showlegend=False), row=1, col=2)
fig.update_layout(barmode="overlay", template="plotly_white", height=430)
fig.show()

---
## Parallel Coordinates

In [5]:
fig = px.parallel_coordinates(
    sites, color="richness",
    dimensions=["elevation","nitrate","phosphorus","ph","richness"],
    color_continuous_scale=px.colors.sequential.Viridis,
    title="Parallel Coordinates: All Numeric Variables"
)
fig.show()
# Drag axes to reorder; click-drag on any axis to filter to a range

---

## Common Pitfalls

**1. Using `fig.show()` in headless environments**  
Use `fig.write_html()` or `fig.write_image()` (requires `kaleido`) for scripts and CI.

**2. Rendering too many points**  
With n > 10,000, use `render_mode='webgl'` or downsample.

**3. Inconsistent `template`**  
Set `template='plotly_white'` consistently; the default grey background rarely suits publication.

**4. Using `make_subplots` when px faceting suffices**  
For standard grids, px `col=`/`row=` parameters are cleaner and share axes automatically.

**5. Sharing as static images**  
Exporting to PNG discards all interactivity. Share as `.html` or embed in notebooks.


---
*python_methods_library · Samantha McGarrigle · [github.com/samantha-mcgarrigle](https://github.com/samantha-mcgarrigle)*